# R4 · NLI fine-tuning (Rathi)

**Runtime → Change runtime type → A100 GPU** (L4 or T4 also work, slower), then **Runtime → Run all**. Run it in one go: the upload cell needs the model trained in the same session.

Uses the same Colab Secrets as R2/R3: `GITHUB_TOKEN`, `HF_TOKEN`, `WANDB_KEY`. Switch on **Notebook access** for all three in the 🔑 panel of *this* notebook first, and click **Grant access** if asked.

Steps: 1 setup → 2 tests + 10-row checks → 3 weak pairs → 4 fine-tune mDeBERTa → 5 score every file → 6 upload model → 7 push → 8 XLM-R-large comparison (optional, ~40 min) → 9 push comparison.

In [4]:
# 1. Setup. The repo is public, so cloning needs no token; GITHUB_TOKEN is only used to push.
from google.colab import userdata
import os, json
REPO = '/content/twincheck'
if not os.path.exists(REPO):
    !git clone https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
!git fetch -q origin && git checkout -q rathi/R4-nli && git pull -q origin rathi/R4-nli

GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
ok = !git ls-remote https://{GH}@github.com/himaallu/Group-Project.git HEAD
print('GitHub token OK' if ok and 'fatal' not in ' '.join(ok) else 'GITHUB_TOKEN rejected: make a new one with the "repo" box ticked')

!pip install -q -U wandb
import wandb
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
wandb.login(key=userdata.get('WANDB_KEY').strip(), relogin=True)
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
HUB_ID = f"{api.whoami()['name']}/twincheck-nli-v1"
print('model will go to', HUB_ID)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Cloning into '/content/twincheck'...
remote: Enumerating objects: 1061, done.
remote: Counting objects: 100% (527/527), done.
remote: Compressing objects: 100% (239/239), done.
remote: Total 1061 (delta 363), reused 426 (delta 285), pack-reused 534 (from 1)
Receiving objects: 100% (1061/1061), 46.20 MiB | 19.35 MiB/s, done.
Resolving deltas: 100% (553/553), done.
/content/twincheck
GitHub token OK
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: rathiabeth (rathiabeth-uowd) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


model will go to RathiAbeth/twincheck-nli-v1
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


## 2. Tests + 10-row checks (pairs, a tiny training run, scoring)

In [5]:
!python -m pytest tests/nli -q
!python -m src.nli.build_pairs --n-per-class 10 --n-val-per-class 10 --out data/nli/test10/train.jsonl     --val-out data/nli/test10/val_weak.jsonl --report data/nli/test10/report.json
!head -n 3 data/nli/test10/train.jsonl
!python -m src.nli.train --train data/nli/test10/train.jsonl --val data/nli/test10/val_weak.jsonl     --n-train 30 --out /content/test10_nli --metrics /content/test10_eval.json     --ckpt-dir /content/ckpt_test10 --no-wandb
!python -m src.nli.score data/processed/en/multifc_val.jsonl --limit 10     --model /content/test10_nli --out-dir /content/test10_scores
import pandas as pd
print(pd.read_parquet('/content/test10_scores/multifc_val_first10.parquet').head())

..................                                                       [100%]
18 passed in 0.11s
{
  "train": {
    "n_pairs": 40,
    "per_label": {
      "neutral": 13,
      "entail": 12,
      "contradict": 15
    },
    "per_language": {
      "en": 30,
      "ar": 10
    },
    "per_origin": {
      "en_random": 10,
      "en_supported": 10,
      "xfact_weak": 8,
      "en_swap_B": 10,
      "multifc_translated": 2
    }
  },
  "val_weak": {
    "n_pairs": 30,
    "per_label": {
      "neutral": 10,
      "contradict": 10,
      "entail": 10
    },
    "per_language": {
      "en": 30
    },
    "per_origin": {
      "en_random": 10,
      "en_contradicted": 10,
      "en_supported": 10
    }
  },
  "arabic_file": "data/nli/ar_train.jsonl",
  "seed": 42,
  "relevance_min": 0.3,
  "per_claim": 2
}
{"id": "multifc_018713_x_multifc_004579_s4", "claim_id": "multifc_018713", "snippet_idx": 4, "language": "en", "claim": "We have been in Afghanistan longer now than we have been in an

## 3. Weak training pairs (~30k English, balanced, + Arabic from Youssef)

In [6]:
!python -m src.nli.build_pairs
print(open('results/R4/pairs_report.json').read())

{
  "train": {
    "n_pairs": 32658,
    "per_label": {
      "entail": 10886,
      "neutral": 10886,
      "contradict": 10886
    },
    "per_language": {
      "en": 30000,
      "ar": 2658
    },
    "per_origin": {
      "en_supported": 10000,
      "en_random": 10000,
      "xfact_weak": 1818,
      "en_swap_B": 3226,
      "multifc_translated": 840,
      "en_contradicted": 6774
    }
  },
  "val_weak": {
    "n_pairs": 1500,
    "per_label": {
      "contradict": 500,
      "neutral": 500,
      "entail": 500
    },
    "per_language": {
      "en": 1500
    },
    "per_origin": {
      "en_contradicted": 500,
      "en_random": 500,
      "en_supported": 500
    }
  },
  "arabic_file": "data/nli/ar_train.jsonl",
  "seed": 42,
  "relevance_min": 0.3,
  "per_claim": 2
}
{
  "train": {
    "n_pairs": 32658,
    "per_label": {
      "entail": 10886,
      "neutral": 10886,
      "contradict": 10886
    },
    "per_language": {
      "en": 30000,
      "ar": 2658
    },
    "per_o

## 4. Fine-tune mDeBERTa-v3 (2 epochs, lr 2e-5, batch 16, fp32/TF32, seed 42; ~20-25 min on an A100)

In [7]:
!python -m src.nli.train
ev = json.load(open('results/R4/gold_eval.json'))
for when in ('zero_shot', 'fine_tuned'):
    print(when, {k: v['macro_f1'] for k, v in ev[when].items()})
print('W&B:', ev['wandb_url'])

32658 train, 1500 weak val, 300 gold pairs; device cuda, fp32
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100% 202/202 [00:00<00:00, 5390.18it/s]
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /root/.netrc.
wandb: Currently logged in as: rathiabeth (rathiabeth-uowd) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.31.0
wandb: Run data is saved locally in /content/twincheck/wandb/run-20261118_035424-gepd51e6
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run R4-nli-mDeBERTa-v3-base-xnli-multilingual-nl

## 5. Score every (claim, snippet) pair with the fine-tuned model → `features/nli_v1/`

In [8]:
FILES = ' '.join([f'data/processed/en/{d}_{s}.jsonl' for d in ('multifc', 'averitec') for s in ('train', 'val', 'test')]
                 + ['data/copies/en/test.jsonl', 'data/copies/en/train.jsonl']
                 + [f'data/processed/ar/xfact_ar_{s}.jsonl' for s in ('train', 'val', 'test')]
                 + ['data/processed/ar/m4fc_ar_test.jsonl'])
!python -m src.nli.score {FILES} --model models/nli_v1 --out-dir features/nli_v1
!du -h features/nli_v1/*

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100% 202/202 [00:00<00:00, 6140.16it/s]
data/processed/en/multifc_train.jsonl: 16847 records, 123233 pairs -> features/nli_v1/multifc_train.parquet (376s)
data/processed/en/multifc_val.jsonl: 2104 records, 15364 pairs -> features/nli_v1/multifc_val.parquet (47s)
data/processed/en/multifc_test.jsonl: 2106 records, 15366 pairs -> features/nli_v1/multifc_test.parquet (45s)
data/processed/en/averitec_train.jsonl: 2309 records, 5675 pairs -> features/nli_v1/averitec_train.parquet (16s)
data/processed/en/averitec_val.jsonl: 252 records, 625 pairs -> features/nli_v1/averitec_val.parquet (2s)
data/processed/en/averitec_test.jsonl: 429 records, 1112 pairs -> features/nli_v1/averitec_test.parquet (3s)
data/copies/en/test.jsonl: 955 records, 6647 pairs -> features/nli_v1/test.parquet (20s)
data/cop

## 6. Upload the model to Hugging Face Hub (too big for the repo's LFS quota) and write a pointer README

In [9]:
api.create_repo(HUB_ID, exist_ok=True, private=False)
api.upload_folder(folder_path='models/nli_v1', repo_id=HUB_ID,
                  commit_message='R4: mDeBERTa-v3 XNLI fine-tuned on TwinCheck weak NLI pairs')
ev = json.load(open('results/R4/gold_eval.json'))
f1 = lambda when, key: ev[when].get(key, {}).get('macro_f1')
lines = [
    '# nli_v1 (R4)',
    '',
    f'Weights are on Hugging Face Hub: https://huggingface.co/{HUB_ID} (too big for git).',
    '',
    f'Load: `from src.nli.score import score_pairs` (finds the Hub copy by itself), or '
    f'`python -m src.nli.score FILE --model {HUB_ID}`.',
    '',
    f"Base `{ev['base_model']}`, {ev['epochs']} epochs, {ev['n_train_pairs']} pairs, lr {ev['lr']}, "
    f"seed {ev['seed']}, {ev['precision']}. Gold macro-F1 EN {f1('zero_shot', 'gold_en')} -> "
    f"{f1('fine_tuned', 'gold_en')}, AR {f1('zero_shot', 'gold_ar')} -> {f1('fine_tuned', 'gold_ar')}. "
    f"W&B: {ev['wandb_url']}",
]
open('models/nli_v1/README.md', 'w').write('\n'.join(lines) + '\n')
print(open('models/nli_v1/README.md').read())

# nli_v1 (R4)

Weights are on Hugging Face Hub: https://huggingface.co/RathiAbeth/twincheck-nli-v1 (too big for git).

Load: `from src.nli.score import score_pairs` (finds the Hub copy by itself), or `python -m src.nli.score FILE --model RathiAbeth/twincheck-nli-v1`.

Base `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`, 2 epochs, 32658 pairs, lr 2e-05, seed 42, fp32 (tf32). Gold macro-F1 EN 0.3711 -> 0.1236, AR 0.4427 -> 0.1617. W&B: https://forge.coreweave.com/wandb/rathiabeth-uowd/twincheck/runs/gepd51e6



## 7. Push the outputs to `rathi/R4-nli`

In [10]:
!git add data/nli/train.jsonl data/nli/val_weak.jsonl features/nli_v1 models/nli_v1/README.md     results/R4/pairs_report.json results/R4/gold_eval.json
!git commit -q -m "[R4] Add weak NLI pairs, fine-tuned NLI scores and gold eval from Colab run"
!git push -q origin rathi/R4-nli && echo PUSHED

PUSHED


## 8. XLM-R-large comparison (optional, ~40 min on an A100). Same data and settings, lr 1e-5 (large models are unstable at 2e-5). Only the gold numbers are kept.

In [11]:
!python -m src.nli.train --model joeddav/xlm-roberta-large-xnli --lr 1e-5     --out /content/nli_xlmr --metrics results/R4/gold_eval_xlmr.json --ckpt-dir /content/ckpt_xlmr
xl = json.load(open('results/R4/gold_eval_xlmr.json'))
for when in ('zero_shot', 'fine_tuned'):
    print(when, {k: v['macro_f1'] for k, v in xl[when].items()})

32658 train, 1500 weak val, 300 gold pairs; device cuda, fp32
config.json: 100% 734/734 [00:00<00:00, 3.04MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 115kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 41.1MB/s]
special_tokens_map.json: 100% 150/150 [00:00<00:00, 789kB/s]

model.safetensors: downloading bytes:  21% 470M/2.24G [00:02<00:04, 430MB/s, 37.6MB/s  ]
model.safetensors: downloading bytes:  25% 562M/2.24G [00:02<00:03, 442MB/s, 45.8MB/s  ]
model.safetensors: downloading bytes:  33% 748M/2.24G [00:02<00:02, 637MB/s, 59.4MB/s  ]
model.safetensors: downloading bytes:  37% 824M/2.24G [00:03<00:02, 531MB/s, 71.0MB/s  ]
model.safetensors: downloading bytes:  72% 1.63G/2.24G [00:04<00:00, 806MB/s,  130MB/s  ]
model.safetensors: downloading bytes:  91% 2.05G/2.24G [00:04<00:00, 814MB/s,  164MB/s  ]
model.safetensors: reconstructing file:  75% 1.68G/2.24G [00:04<00:01, 510MB/s,  108MB/s  ]
model.safetensors: reconstructing file:  90% 2.01G/2.24G [00:07<00:00, 

## 9. Push the comparison

In [12]:
!git add results/R4/gold_eval_xlmr.json
!git commit -q -m "[R4] Add XLM-R-large comparison on the gold pairs"
!git push -q origin rathi/R4-nli && echo PUSHED

PUSHED


**Done.** Tell Claude it finished. Then **Runtime → Disconnect and delete runtime**.